# Medical Text Classification for Symptom Analysis

## Research Context
**Title:** NLP and Deep Learning for Text and Audio Classification in Medical Diagnosis

**Research Question 1 (RQ1):** What is the effectiveness of the NLP algorithm in classifying patient symptoms from the text data on the population level?

**Hypotheses:**
- **H10 (Null):** Text analysis of patient symptoms results in insufficient precision and recall for provider decision support.
- **H1a (Alternative):** Text analysis of patient symptoms results in precision and recall sufficient for provider decision support.

This notebook implements comprehensive text classification techniques to address RQ1 and test the associated hypotheses.

## Step 1: Problem Definition

This notebook addresses the research: **NLP and Deep Learning for Text and Audio Classification in Medical Diagnosis**.

## Research Questions
- **RQ1:** What is the effectiveness of the NLP algorithm in classifying patient symptoms from the text data on the population level?
- **RQ2:** How effective is NLP in the classifying of patient symptoms from audio data?

## Hypotheses
- **H10:** Text analysis of patient symptoms results in insufficient precision and recall for provider decision support.
- **H1a:** Text analysis of patient symptoms results in precision and recall sufficient for provider decision support.
- **H20:** Audio analysis of patient symptoms yields both precision and recall metrics that are insufficient for effective provider decision support.
- **H2a:** Audio analysis of patient symptoms results in precision and recall sufficient for provider decision support.

In [ ]:
# Data manipulation and analysis
import pandas as pd              # For dataframe manipulation and analysis
import numpy as np              # For numerical operations and matrix handling
import os                       # For file path operations 
import json                     # For handling JSON data
from datetime import datetime   # For timestamp operations

# Text preprocessing and NLP libraries
import re                       # For regular expressions in text cleaning
import nltk                     # Natural Language Toolkit for text processing
from nltk.corpus import stopwords           # For removing common words
from nltk.tokenize import word_tokenize     # For splitting text into words
from nltk.stem import WordNetLemmatizer     # For reducing words to base form
from nltk.util import ngrams                # For creating n-grams features
import string                   # For string operations (punctuation removal)

# Advanced NLP capabilities
import spacy                    # For advanced NLP functionality
from textblob import TextBlob   # For sentiment analysis
from gensim.models import Word2Vec          # For word embeddings
from sklearn.feature_extraction.text import (
    CountVectorizer,            # For Bag of Words vectorization
    TfidfVectorizer,            # For TF-IDF vectorization
    HashingVectorizer           # For hashing-based vectorization
)

# Machine Learning - Model Selection and Validation
from sklearn.model_selection import (
    train_test_split,           # For splitting data into train/test sets
    cross_val_score,            # For k-fold cross-validation
    GridSearchCV,               # For hyperparameter tuning
    StratifiedKFold,            # For stratified k-fold cross-validation
    learning_curve,             # For generating learning curves
    validation_curve            # For validation curves
)
from sklearn.preprocessing import (
    LabelEncoder,               # For encoding categorical variables
    StandardScaler,             # For standardizing features
    MinMaxScaler                # For scaling features to a range
)
from sklearn.pipeline import Pipeline      # For creating ML pipelines
from sklearn.compose import ColumnTransformer  # For applying transformers to columns

# Traditional Machine Learning Models
from sklearn.linear_model import (
    LogisticRegression,         # For logistic regression
    SGDClassifier               # For stochastic gradient descent classifier
)
from sklearn.naive_bayes import (
    MultinomialNB,              # For multinomial naive bayes
    ComplementNB                # For complement naive bayes
)
from sklearn.svm import (
    SVC,                        # For support vector classification
    LinearSVC                   # For linear support vector classification
)
from sklearn.ensemble import (
    RandomForestClassifier,     # For random forest ensemble
    GradientBoostingClassifier, # For gradient boosting
    VotingClassifier,           # For ensemble voting
    AdaBoostClassifier          # For AdaBoost ensemble
)
from sklearn.tree import DecisionTreeClassifier  # For decision trees
from sklearn.neighbors import KNeighborsClassifier  # For k-nearest neighbors

# Deep Learning
import tensorflow as tf        # Core TensorFlow library
from tensorflow.keras.preprocessing.text import Tokenizer  # For text tokenization
from tensorflow.keras.preprocessing.sequence import pad_sequences  # For padding sequences
from tensorflow.keras.models import (
    Sequential,                # For building sequential models
    Model,                     # For building functional API models
    load_model                 # For loading saved models
)
from tensorflow.keras.layers import (
    Embedding,                 # For word embeddings
    GlobalAveragePooling1D,    # For pooling operations
    GlobalMaxPooling1D,        # For max pooling
    Conv1D,                    # For 1D convolutions
    LSTM,                      # For long short-term memory cells
    Bidirectional,             # For bidirectional RNNs
    Dense,                     # For fully connected layers
    Dropout,                   # For regularization
    BatchNormalization,        # For batch normalization
    Attention,                 # For attention mechanism (if available)
    LayerNormalization         # For layer normalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,             # For early stopping
    ModelCheckpoint,           # For saving best models
    ReduceLROnPlateau          # For learning rate scheduling
)
from tensorflow.keras.optimizers import (
    Adam,                      # Adam optimizer
    RMSprop                    # RMSprop optimizer
)

# Model Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,            # For classification accuracy
    precision_score,           # For precision metric
    recall_score,              # For recall metric
    f1_score,                  # For F1 score
    confusion_matrix,          # For confusion matrix
    classification_report,     # For detailed classification metrics
    roc_curve,                 # For ROC curve
    auc,                       # For AUC calculation
    roc_auc_score,             # For ROC AUC score
    precision_recall_curve,    # For precision-recall curve
    average_precision_score    # For average precision
)

# Visualization
import matplotlib.pyplot as plt                # For basic plotting
import seaborn as sns                          # For statistical data visualization
from wordcloud import WordCloud                # For word cloud visualization
import plotly.express as px                    # For interactive plots
import plotly.graph_objects as go              # For custom plotly visualizations
from plotly.subplots import make_subplots      # For subplot creation

# Set random seeds for reproducibility across all libraries
np.random.seed(42)             # NumPy seed
tf.random.set_seed(42)         # TensorFlow seed
import random
random.seed(42)                # Python's random module seed

# Download necessary NLTK resources
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')

# Initialize spaCy model - for advanced NLP features 
# Uncomment if needed (larger model):
# try:
#     nlp = spacy.load('en_core_web_md')  # Medium-sized model
# except:
#     import sys
#     !{sys.executable} -m spacy download en_core_web_md
#     nlp = spacy.load('en_core_web_md')

# Load smaller spaCy model for efficient processing
try:
    nlp = spacy.load('en_core_web_sm')  # Small model for efficiency
except:
    import sys
    os.system(f"{sys.executable} -m spacy download en_core_web_sm")
    nlp = spacy.load('en_core_web_sm')

# Suppress warning messages for cleaner notebook output
import warnings
warnings.filterwarnings('ignore')

# Log session start for reproducibility
print(f"Environment setup complete at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"TensorFlow version: {tf.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"spaCy version: {spacy.__version__}")

## Step 2: Data Loading and Initial Inspection

This step involves loading the medical symptom dataset from the specified path and performing initial inspection to understand its structure, content, and quality. We'll focus specifically on the "phrase" and "prompt" fields for our text classification task.

In [ ]:
# Define base paths for the dataset
BASE_PATH = r"G:\Msc\NCU\Doctoral Record\multimodal_medical_diagnosis\data\Medical Speech, Transcription, and Intent"

# Define paths for different dataset components
PATHS = {
    'csv': os.path.join(BASE_PATH, "overview-of-recordings.csv"),
    'test_audio': os.path.join(BASE_PATH, "recordings", "test"),
    'train_audio': os.path.join(BASE_PATH, "recordings", "train"),
    'validate_audio': os.path.join(BASE_PATH, "recordings", "validate")
}

def load_medical_dataset(csv_path):
    """
    Load medical text dataset from CSV file and perform initial data inspection
    
    Args:
        csv_path (str): Path to the CSV file containing medical symptom data
    
    Returns:
        pd.DataFrame: Loaded and initially preprocessed dataset
    """
    try:
        # Load dataset using pandas
        print(f"Loading dataset from: {csv_path}")
        df = pd.read_csv(csv_path)
        
        # Display initial information about the dataset
        print(f"\nInitial Dataset Information:")
        print(f"- Total number of records: {df.shape[0]}")
        print(f"- Number of features: {df.shape[1]}")
        
        # Check column names and standardize if needed
        print(f"\nOriginal column names: {list(df.columns)}")
        
        # Standardize column names to lowercase for consistency
        df.columns = df.columns.str.lower().str.strip()
        print(f"Standardized column names: {list(df.columns)}")
        
        # Check for presence of key fields (phrase and prompt)
        key_fields = ['phrase', 'prompt']
        missing_fields = [field for field in key_fields if field not in df.columns]
        
        if missing_fields:
            print(f"WARNING: The following key fields are missing: {missing_fields}")
            return None
        
        # Check for missing values in key fields
        missing_values = df[key_fields].isnull().sum()
        print(f"\nMissing values in key fields:")
        for field, count in missing_values.items():
            print(f"- {field}: {count} missing values ({count/len(df)*100:.2f}%)")
        
        # Remove rows with missing values in key fields
        original_count = len(df)
        df = df.dropna(subset=key_fields)
        removed_count = original_count - len(df)
        print(f"\nRemoved {removed_count} rows with missing values in key fields ({removed_count/original_count*100:.2f}% of data)")
        
        # Display basic statistics about the phrase length
        df['phrase_length'] = df['phrase'].str.len()
        df['word_count'] = df['phrase'].str.split().str.len()
        
        print(f"\nPhrase statistics:")
        print(f"- Average phrase length: {df['phrase_length'].mean():.2f} characters")
        print(f"- Average word count: {df['word_count'].mean():.2f} words")
        print(f"- Minimum length: {df['phrase_length'].min()} characters")
        print(f"- Maximum length: {df['phrase_length'].max()} characters")
        
        # Display information about classes (prompts)
        unique_prompts = df['prompt'].nunique()
        print(f"\nClass information:")
        print(f"- Number of unique prompts (classes): {unique_prompts}")
        
        # Check class distribution
        class_counts = df['prompt'].value_counts()
        max_class_size = class_counts.max()
        min_class_size = class_counts.min()
        imbalance_ratio = max_class_size / min_class_size
        
        print(f"- Most frequent prompt: '{class_counts.index[0]}' with {max_class_size} samples")
        print(f"- Least frequent prompt: '{class_counts.index[-1]}' with {min_class_size} samples")
        print(f"- Class imbalance ratio: {imbalance_ratio:.2f}")
        
        # Check for potential audio associations (for future work on RQ2)
        if 'file_name' in df.columns:
            audio_files = df['file_name'].notna().sum()
            print(f"\nAudio association:")
            print(f"- {audio_files} records ({audio_files/len(df)*100:.2f}%) have associated audio files")
        
        return df
    
    except Exception as e:
        print(f"Error loading dataset: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Load the dataset
df = load_medical_dataset(PATHS['csv'])

# Display sample data to inspect the format
print("\n--- Sample Data (First 5 Records) ---")
display(df.head())

In [ ]:
# Function to perform more detailed data inspection
def inspect_dataset(df):
    """
    Perform detailed inspection of the dataset with visualizations
    
    Args:
        df (pd.DataFrame): The loaded medical dataset
        
    Returns:
        None: Outputs visualizations and statistics directly
    """
    # Create a figure for initial visualizations
    plt.figure(figsize=(16, 14))
    
    # 1. Visualize class distribution (top N classes)
    plt.subplot(2, 2, 1)
    top_n = 10
    top_prompts = df['prompt'].value_counts().head(top_n)
    sns.barplot(y=top_prompts.index, x=top_prompts.values, palette='viridis')
    plt.title(f'Distribution of Top {top_n} Medical Conditions (Prompts)', fontsize=12)
    plt.xlabel('Number of Samples', fontsize=10)
    plt.ylabel('Medical Condition', fontsize=10)
    plt.tight_layout()
    
    # 2. Text length distribution
    plt.subplot(2, 2, 2)
    sns.histplot(df['phrase_length'], bins=30, kde=True)
    plt.axvline(df['phrase_length'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["phrase_length"].mean():.1f}')
    plt.title('Distribution of Phrase Lengths (Characters)', fontsize=12)
    plt.xlabel('Number of Characters', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    
    # 3. Word count distribution
    plt.subplot(2, 2, 3)
    sns.histplot(df['word_count'], bins=30, kde=True)
    plt.axvline(df['word_count'].mean(), color='red', linestyle='--', 
                label=f'Mean: {df["word_count"].mean():.1f}')
    plt.title('Distribution of Word Counts per Phrase', fontsize=12)
    plt.xlabel('Number of Words', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    
    # 4. Box plot of phrase length by top prompts
    plt.subplot(2, 2, 4)
    top_5_prompts = df['prompt'].value_counts().head(5).index
    sns.boxplot(x='word_count', y='prompt', data=df[df['prompt'].isin(top_5_prompts)], 
                palette='Set2', orient='h')
    plt.title('Word Count Distribution by Top 5 Medical Conditions', fontsize=12)
    plt.xlabel('Number of Words', fontsize=10)
    plt.ylabel('Medical Condition', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # 5. Check for duplicates in the dataset
    duplicates = df.duplicated(subset=['phrase']).sum()
    print(f"\nDuplicate Analysis:")
    print(f"- Number of duplicate phrases: {duplicates} ({duplicates/len(df)*100:.2f}% of data)")
    
    # 6. Generate quick statistics by prompt
    prompt_stats = df.groupby('prompt').agg({
        'phrase_length': ['mean', 'min', 'max', 'std'],
        'word_count': ['mean', 'min', 'max', 'std'],
        'phrase': 'count'
    }).sort_values(('phrase', 'count'), ascending=False)
    
    print("\n--- Prompt Statistics (Top 5 Classes) ---")
    display(prompt_stats.head(5))
    
    return None

# Execute the inspection
inspect_dataset(df)

## Step 3: Data Cleaning and Preprocessing

We will load the dataset, clean the text, and engineer features for modeling. All code is thoroughly commented for clarity.

In [ ]:
def preprocess_text(text, remove_stopwords=True, lemmatize=True, 
                    remove_numbers=True, remove_punctuation=True):
    """
    Comprehensive text preprocessing function for medical symptom text
    
    Args:
        text (str): Input medical symptom text to preprocess
        remove_stopwords (bool): Whether to remove stopwords
        lemmatize (bool): Whether to apply lemmatization
        remove_numbers (bool): Whether to remove numerical characters
        remove_punctuation (bool): Whether to remove punctuation
    
    Returns:
        str: Cleaned and preprocessed text
    """
    # Ensure text is string (handle potential NaN values)
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase for consistency
    text = text.lower()
    
    # Remove URLs if any present in text
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove email addresses if any
    text = re.sub(r'\S*@\S*\s?', '', text)
    
    # Remove punctuation if specified
    if remove_punctuation:
        text = re.sub(r'[^\w\s]', ' ', text)
    
    # Remove numbers if specified 
    if remove_numbers:
        text = re.sub(r'\d+', '', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize text into individual words
    tokens = word_tokenize(text)
    
    # Remove stopwords if specified
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        # Keep some important medical stopwords that might be relevant
        medical_relevant_words = {'no', 'not', 'nor', 'pain', 'ache'}
        filtered_stop_words = stop_words - medical_relevant_words
        tokens = [token for token in tokens if token not in filtered_stop_words]
    
    # Apply lemmatization if specified (reduce words to base form)
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Rejoin tokens into a single string
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

# Create a function to add advanced text features
def add_spacy_features(text):
    """
    Extract advanced linguistic features using spaCy
    
    Args:
        text (str): Input text
    
    Returns:
        dict: Dictionary containing advanced linguistic features
    """
    # Process text with spaCy
    doc = nlp(text)
    
    # Extract features
    features = {
        'num_entities': len(doc.ents),                      # Named entities
        'sentiment': TextBlob(text).sentiment.polarity,     # Text sentiment
        'num_sentences': len(list(doc.sents)),              # Number of sentences
        'contains_negation': any(token.dep_ == 'neg' for token in doc),  # Negation presence
        'avg_token_length': np.mean([len(token.text) for token in doc]) if len(doc) > 0 else 0,
    }
    
    return features

# Apply text preprocessing to the dataset
print("Applying text preprocessing...")
start_time = datetime.now()

# Create variations of processed text for experimentation
df['processed_text_full'] = df['phrase'].apply(
    lambda x: preprocess_text(x, remove_stopwords=False, lemmatize=False))

df['processed_text'] = df['phrase'].apply(preprocess_text)

df['processed_text_minimal'] = df['phrase'].apply(
    lambda x: preprocess_text(x, remove_stopwords=True, lemmatize=True))

print(f"Text preprocessing completed in {(datetime.now() - start_time).total_seconds():.2f} seconds")

# Display sample texts before and after preprocessing
sample_size = min(5, len(df))
sample_indices = np.random.choice(len(df), sample_size, replace=False)

print("\n--- Sample Texts Before and After Preprocessing ---")
for i in sample_indices:
    print(f"\nOriginal: \"{df['phrase'].iloc[i]}\"")
    print(f"Processed (full): \"{df['processed_text_full'].iloc[i]}\"")
    print(f"Processed (standard): \"{df['processed_text'].iloc[i]}\"")
    print(f"Processed (minimal): \"{df['processed_text_minimal'].iloc[i]}\"")
    print("-" * 50)

# Add advanced linguistic features using spaCy (sampling for efficiency)
print("\nExtracting advanced linguistic features (on a small sample for demonstration)...")

# Apply to a smaller sample for demonstration purposes (to prevent long execution time)
sample_size = min(100, len(df))
sample_indices = np.random.choice(len(df), sample_size, replace=False)
sample_df = df.iloc[sample_indices].copy()

# Extract linguistic features
spacy_features = sample_df['processed_text'].apply(add_spacy_features)

# Convert list of dictionaries to DataFrame columns
for feature in ['num_entities', 'sentiment', 'num_sentences', 'contains_negation', 'avg_token_length']:
    sample_df[feature] = spacy_features.apply(lambda x: x[feature])

print("Linguistic feature extraction completed")
print("\n--- Sample with Linguistic Features ---")
display(sample_df[['phrase', 'prompt', 'num_entities', 'sentiment', 'contains_negation']].head(3))

## Step 4: Exploratory Data Analysis (EDA)

We will explore the dataset to understand its structure, class balance, and text characteristics. Visualizations and statistics are provided for research transparency.

In [ ]:
def perform_advanced_text_eda(df):
    """
    Perform comprehensive text-based exploratory data analysis aligned with research questions
    
    Args:
        df (pd.DataFrame): Preprocessed dataset containing medical symptoms text
    """
    print("Performing advanced exploratory data analysis...")
    print("\n1. Text Length Characteristics")
    
    # 1. Text length analysis
    fig = plt.figure(figsize=(16, 12))
    
    # Text length distribution by characters
    plt.subplot(2, 2, 1)
    sns.histplot(data=df, x='phrase_length', kde=True, bins=30)
    plt.axvline(df['phrase_length'].mean(), color='r', linestyle='--', 
               label=f'Mean: {df["phrase_length"].mean():.1f}')
    plt.title('Distribution of Phrase Lengths (Characters)', fontsize=12)
    plt.xlabel('Number of Characters', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    
    # Word count distribution
    plt.subplot(2, 2, 2)
    sns.histplot(data=df, x='word_count', kde=True, bins=30)
    plt.axvline(df['word_count'].mean(), color='r', linestyle='--', 
               label=f'Mean: {df["word_count"].mean():.1f}')
    plt.title('Distribution of Word Counts', fontsize=12)
    plt.xlabel('Word Count', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    
    # Word count distribution by top medical conditions
    plt.subplot(2, 2, 3)
    top_n = 5
    top_conditions = df['prompt'].value_counts().head(top_n).index
    sns.boxplot(x='word_count', y='prompt', data=df[df['prompt'].isin(top_conditions)], 
               palette='viridis')
    plt.title(f'Word Count by Top {top_n} Medical Conditions', fontsize=12)
    plt.xlabel('Word Count', fontsize=10)
    plt.ylabel('Medical Condition', fontsize=10)
    
    # Processed text length distribution
    plt.subplot(2, 2, 4)
    df['processed_length'] = df['processed_text'].apply(len)
    sns.histplot(data=df, x='processed_length', kde=True, bins=30, color='green')
    plt.axvline(df['processed_length'].mean(), color='r', linestyle='--', 
               label=f'Mean: {df["processed_length"].mean():.1f}')
    plt.title('Processed Text Length Distribution', fontsize=12)
    plt.xlabel('Processed Text Length (Characters)', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 2. Word frequency analysis
    print("\n2. Word Frequency Analysis")
    
    # Create word frequency counters
    all_words = ' '.join(df['processed_text']).split()
    word_freq = Counter(all_words)
    
    # Plot top words
    plt.figure(figsize=(12, 6))
    top_words = dict(word_freq.most_common(20))
    plt.bar(top_words.keys(), top_words.values(), color='skyblue')
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 20 Most Frequent Words in Medical Symptoms', fontsize=12)
    plt.xlabel('Word', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.tight_layout()
    plt.show()
    
    # Generate word cloud
    plt.figure(figsize=(12, 8))
    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        max_words=100,
        contour_width=3,
        contour_color='steelblue'
    ).generate(' '.join(df['processed_text']))
    
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Medical Symptoms', fontsize=14)
    plt.show()
    
    # 3. Class distribution analysis
    print("\n3. Medical Condition (Prompt) Distribution Analysis")
    
    # Calculate class distribution
    class_dist = df['prompt'].value_counts()
    total_classes = len(class_dist)
    
    # Calculate class imbalance metrics
    min_class_size = class_dist.min()
    max_class_size = class_dist.max()
    avg_class_size = class_dist.mean()
    imbalance_ratio = max_class_size / min_class_size
    
    print(f"Class Distribution Stats:")
    print(f"- Total medical conditions (classes): {total_classes}")
    print(f"- Most frequent: '{class_dist.index[0]}' with {max_class_size} samples")
    print(f"- Least frequent: '{class_dist.index[-1]}' with {min_class_size} samples")
    print(f"- Average samples per class: {avg_class_size:.2f}")
    print(f"- Class imbalance ratio (max/min): {imbalance_ratio:.2f}")
    
    # Plot class distribution
    plt.figure(figsize=(14, 6))
    class_dist_percent = 100 * class_dist / class_dist.sum()
    
    plt.subplot(1, 2, 1)
    sns.barplot(y=class_dist.index[:10], x=class_dist.values[:10], palette='viridis')
    plt.title('Top 10 Medical Conditions by Frequency', fontsize=12)
    plt.xlabel('Number of Samples', fontsize=10)
    plt.ylabel('Medical Condition', fontsize=10)
    
    plt.subplot(1, 2, 2)
    plt.pie(class_dist_percent[:5], labels=class_dist.index[:5], autopct='%1.1f%%',
           shadow=True, startangle=90)
    plt.axis('equal')
    plt.title('Top 5 Medical Conditions (% of Dataset)', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # 4. Word usage by medical condition (prompt)
    print("\n4. Word Usage by Medical Condition")
    
    # Get top conditions for analysis
    top_conditions = class_dist.head(5).index.tolist()
    
    # Create comparison word clouds for top conditions
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, condition in enumerate(top_conditions):
        if i >= len(axes):
            break
            
        # Get all text for this condition
        condition_text = ' '.join(df[df['prompt'] == condition]['processed_text'])
        
        # Generate word cloud for this condition
        condition_cloud = WordCloud(
            width=400, height=200,
            background_color='white',
            max_words=50,
            colormap='viridis'
        ).generate(condition_text)
        
        # Plot word cloud
        axes[i].imshow(condition_cloud, interpolation='bilinear')
        axes[i].set_title(f'Word Cloud: {condition}', fontsize=12)
        axes[i].axis('off')
    
    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 5. Relationship between text length and medical condition
    print("\n5. Relationship Between Text Characteristics and Medical Condition")
    
    # Calculate average text metrics by prompt
    prompt_text_stats = df.groupby('prompt').agg({
        'phrase_length': ['mean', 'std'],
        'word_count': ['mean', 'std'],
        'processed_length': ['mean', 'std']
    }).sort_values(('phrase_length', 'mean'), ascending=False)
    
    # Display top conditions by text length
    print("\nMedical conditions with longest symptom descriptions:")
    display(prompt_text_stats.head(5))
    
    print("\nMedical conditions with shortest symptom descriptions:")
    display(prompt_text_stats.tail(5))
    
    # 6. Research question specific analysis: Text classification potential
    print("\n6. RQ1 Analysis: Text Classification Potential")
    
    # Calculate vocabulary diversity by prompt (unique words / total words)
    prompt_vocab = {}
    for prompt in top_conditions:
        # Get all processed texts for this prompt
        texts = df[df['prompt'] == prompt]['processed_text']
        
        # Calculate vocabulary metrics
        all_words = ' '.join(texts).split()
        unique_words = set(all_words)
        
        prompt_vocab[prompt] = {
            'total_words': len(all_words),
            'unique_words': len(unique_words),
            'vocabulary_density': len(unique_words) / len(all_words) if len(all_words) > 0 else 0,
            'avg_phrase_length': texts.str.len().mean()
        }
    
    # Create dataframe for visualization
    vocab_df = pd.DataFrame.from_dict(prompt_vocab, orient='index')
    
    # Plot vocabulary density by prompt
    plt.figure(figsize=(12, 6))
    sns.barplot(x=vocab_df.index, y=vocab_df['vocabulary_density'], palette='plasma')
    plt.title('Vocabulary Density by Medical Condition', fontsize=12)
    plt.xlabel('Medical Condition', fontsize=10)
    plt.ylabel('Vocabulary Density (Unique/Total Words)', fontsize=10)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Display vocabulary stats
    print("\nVocabulary statistics by medical condition:")
    display(vocab_df)
    
    # 7. Evaluation of H10/H1a potential (Text classification potential)
    print("\n7. Hypothesis Evaluation Potential (H10/H1a)")
    
    # Calculate metrics that would suggest good classification potential
    # Higher values indicate better potential for classification
    classification_potential = {
        'vocab_diversity': vocab_df['vocabulary_density'].mean(),  # Higher is better
        'class_balance': 1 / imbalance_ratio,  # Higher is better (1 = perfectly balanced)
        'samples_per_class': avg_class_size,  # Higher is better
        'text_length_consistency': 1 / df['phrase_length'].std()  # Higher means more consistent
    }
    
    # Calculate overall potential score (simple average of normalized metrics)
    # Each metric normalized to 0-1 scale using arbitrary thresholds
    potential_score = (
        min(classification_potential['vocab_diversity'] / 0.5, 1) * 0.25 +
        classification_potential['class_balance'] * 0.25 +
        min(classification_potential['samples_per_class'] / 100, 1) * 0.25 +
        min(classification_potential['text_length_consistency'] * 10, 1) * 0.25
    )
    
    print(f"Classification potential metrics:")
    for metric, value in classification_potential.items():
        print(f"- {metric}: {value:.4f}")
    
    print(f"\nOverall classification potential score: {potential_score:.2f} out of 1.00")
    
    if potential_score > 0.6:
        print("Initial assessment suggests good potential for text classification (leaning toward H1a)")
    else:
        print("Initial assessment suggests challenges for text classification (leaning toward H10)")
    
    print("\nNote: Final hypothesis testing requires actual model training and evaluation")
    
    return {
        'class_count': total_classes,
        'imbalance_ratio': imbalance_ratio,
        'classification_potential': potential_score
    }

# Run the advanced EDA
eda_results = perform_advanced_text_eda(df)

## Step 5: Feature Engineering

This step involves creating advanced features from the text data that will help improve classification performance. We'll engineer various features based on the symptoms text and then combine them with our existing dataset.

### Encoding Class Labels and Feature Normalization

Before model training, we encode the target class ("prompt") into numerical labels and normalize the feature set. This ensures compatibility with machine learning algorithms and improves model performance. All steps are commented for clarity.

In [ ]:
# Encode the target class (prompt) into numerical labels
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode class labels
y = df_with_features['prompt']
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Prepare feature set (drop non-feature columns)
feature_cols = [col for col in df_with_features.columns if col not in ['phrase', 'prompt']]
X = df_with_features[feature_cols].values

# Normalize features for ML algorithms
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Display encoded class mapping and feature shape
print("Class label mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")
print(f"\nFeature matrix shape: {X_normalized.shape}")
print(f"Number of classes: {len(label_encoder.classes_)}")

In [ ]:
def engineer_text_features(df, original_text_col='phrase', processed_text_col='processed_text'):
    """
    Create advanced text features for medical symptom classification
    
    Args:
        df (pd.DataFrame): Input dataframe containing symptom texts
        original_text_col (str): Column name containing original text
        processed_text_col (str): Column name containing processed text
    
    Returns:
        pd.DataFrame: Dataframe with engineered features
    """
    print("Engineering advanced text features...")
    
    # Make a copy to avoid modifying the original dataframe
    df_features = df.copy()
    
    # 1. Basic text statistics features
    print("1. Creating basic text statistics features...")
    
    # Character-level features
    df_features['char_count'] = df_features[original_text_col].str.len()
    df_features['char_count_no_spaces'] = df_features[original_text_col].apply(lambda x: len(x.replace(" ", "")))
    df_features['spaces_count'] = df_features[original_text_col].apply(lambda x: x.count(' '))
    df_features['uppercase_char_count'] = df_features[original_text_col].apply(lambda x: sum(1 for c in x if c.isupper()))
    df_features['lowercase_char_count'] = df_features[original_text_col].apply(lambda x: sum(1 for c in x if c.islower()))
    df_features['digit_count'] = df_features[original_text_col].apply(lambda x: sum(c.isdigit() for c in x))
    df_features['punctuation_count'] = df_features[original_text_col].apply(lambda x: sum(c in string.punctuation for c in x))
    
    # Word-level features
    df_features['word_count'] = df_features[original_text_col].apply(lambda x: len(x.split()))
    df_features['processed_word_count'] = df_features[processed_text_col].apply(lambda x: len(x.split()))
    df_features['avg_word_length'] = df_features[original_text_col].apply(lambda x: 
                                                                        np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)
    
    # 2. Linguistic complexity features
    print("2. Creating linguistic complexity features...")
    
    # Vocabulary richness
    df_features['unique_word_ratio'] = df_features[processed_text_col].apply(
        lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0
    )
    
    # Type-token ratio (another measure of lexical diversity)
    df_features['type_token_ratio'] = df_features[processed_text_col].apply(
        lambda x: len(set(x.split())) / max(1, len(x.split()))
    )
    
    # Hapax legomena ratio (ratio of words appearing only once)
    df_features['hapax_ratio'] = df_features[processed_text_col].apply(
        lambda x: len([word for word, freq in Counter(x.split()).items() if freq == 1]) / 
                 max(1, len(set(x.split())))
    )
    
    # 3. Medical symptom-specific features
    print("3. Creating medical domain-specific features...")
    
    # Define common medical symptom terms and categories
    pain_words = ['pain', 'ache', 'hurt', 'discomfort', 'sore', 'sharp', 'tender']
    severity_words = ['mild', 'moderate', 'severe', 'extreme', 'intense', 'excruciating']
    duration_words = ['chronic', 'acute', 'persistent', 'intermittent', 'constant', 'occasional']
    location_words = ['head', 'chest', 'abdomen', 'arm', 'leg', 'back', 'throat', 'ear', 'eye']
    
    # Count occurrences of each category
    df_features['pain_word_count'] = df_features[processed_text_col].apply(
        lambda x: sum(word in x.split() for word in pain_words)
    )
    
    df_features['severity_word_count'] = df_features[processed_text_col].apply(
        lambda x: sum(word in x.split() for word in severity_words)
    )
    
    df_features['duration_word_count'] = df_features[processed_text_col].apply(
        lambda x: sum(word in x.split() for word in duration_words)
    )
    
    df_features['location_word_count'] = df_features[processed_text_col].apply(
        lambda x: sum(word in x.split() for word in location_words)
    )
    
    # Presence of negation
    negation_words = ['no', 'not', "don't", 'never', 'none', 'neither', 'nor', 'without']
    df_features['contains_negation'] = df_features[processed_text_col].apply(
        lambda x: any(neg in x.split() for neg in negation_words)
    ).astype(int)
    
    # 4. N-gram features (for demonstration - normall we'd use vectorizers for this)
    print("4. Creating n-gram demonstration features...")
    
    # Extract bigrams for top conditions (as examples)
    top_conditions = df['prompt'].value_counts().head(5).index.tolist()
    condition_bigrams = {}
    
    for condition in top_conditions:
        # Get all processed texts for this condition
        condition_texts = df[df['prompt'] == condition][processed_text_col].tolist()
        condition_words = [text.split() for text in condition_texts]
        
        # Extract bigrams
        all_bigrams = []
        for words in condition_words:
            if len(words) > 1:
                all_bigrams.extend(list(ngrams(words, 2)))
        
        # Count bigram frequencies
        bigram_counts = Counter(all_bigrams)
        
        # Store top bigrams for this condition
        condition_bigrams[condition] = dict(bigram_counts.most_common(5))
    
    # Print top bigrams by condition
    print("\nTop bigrams by medical condition:")
    for condition, bigrams in condition_bigrams.items():
        print(f"- {condition}: {[' '.join(bg) for bg in bigrams.keys()]}")
    
    # 5. Create sentiment features using TextBlob
    print("5. Creating sentiment features...")
    
    # Apply sentiment analysis to a sample of 1000 rows maximum (for efficiency)
    sample_size = min(1000, len(df_features))
    random_sample = df_features.sample(sample_size)
    
    # Calculate sentiment scores for the sample
    sample_sentiments = random_sample[original_text_col].apply(
        lambda x: TextBlob(x).sentiment
    )
    
    # Extract polarity and subjectivity
    random_sample['sentiment_polarity'] = sample_sentiments.apply(lambda x: x.polarity)
    random_sample['sentiment_subjectivity'] = sample_sentiments.apply(lambda x: x.subjectivity)
    
    # Visualize sentiment distribution for the sample
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.histplot(data=random_sample, x='sentiment_polarity', kde=True)
    plt.title('Sentiment Polarity Distribution')
    plt.xlabel('Polarity (-1: Negative, +1: Positive)')
    plt.ylabel('Count')
    
    plt.subplot(1, 2, 2)
    sns.histplot(data=random_sample, x='sentiment_subjectivity', kde=True)
    plt.title('Sentiment Subjectivity Distribution')
    plt.xlabel('Subjectivity (0: Objective, 1: Subjective)')
    plt.ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSummary of engineered features:")
    
    # Categories of features
    feature_categories = {
        'Basic Text Statistics': ['char_count', 'char_count_no_spaces', 'spaces_count', 
                                 'uppercase_char_count', 'lowercase_char_count', 
                                 'digit_count', 'punctuation_count', 'word_count', 
                                 'processed_word_count', 'avg_word_length'],
        'Linguistic Complexity': ['unique_word_ratio', 'type_token_ratio', 'hapax_ratio'],
        'Medical Domain-Specific': ['pain_word_count', 'severity_word_count', 
                                   'duration_word_count', 'location_word_count', 
                                   'contains_negation']
    }
    
    for category, features in feature_categories.items():
        print(f"- {category}: {len(features)} features")
    
    print(f"\nTotal engineered features: {sum(len(f) for f in feature_categories.values())}")
    
    return df_features

# Apply feature engineering
df_with_features = engineer_text_features(df)

# Display sample with engineered features
print("\n--- Sample with Engineered Features ---")
# Select subset of columns to display (to keep output manageable)
feature_sample_cols = ['phrase', 'prompt', 'word_count', 'unique_word_ratio', 
                      'pain_word_count', 'severity_word_count', 'contains_negation']
display(df_with_features[feature_sample_cols].head())

# Feature importance analysis for medical conditions
def analyze_feature_importance(df_with_features):
    """
    Analyze which engineered features are most distinctive for different medical conditions
    
    Args:
        df_with_features: DataFrame with engineered features
    """
    print("\nAnalyzing feature importance by medical condition...")
    
    # Select numeric features
    numeric_features = df_with_features.select_dtypes(include=['number']).columns.tolist()
    numeric_features = [f for f in numeric_features if f not in ['phrase_length', 'processed_length']]
    
    # Get top conditions
    top_conditions = df_with_features['prompt'].value_counts().head(5).index.tolist()
    
    # Calculate feature means by condition
    feature_means = df_with_features.groupby('prompt')[numeric_features].mean()
    
    # Create heatmap for top conditions
    plt.figure(figsize=(14, 8))
    sns.heatmap(feature_means.loc[top_conditions, numeric_features[:10]], annot=True, 
                cmap='YlGnBu', fmt='.2f')
    plt.title('Feature Values by Medical Condition (Top 5)')
    plt.xlabel('Engineered Features')
    plt.ylabel('Medical Condition')
    plt.tight_layout()
    plt.show()
    
    return feature_means

# Analyze feature importance
feature_importance = analyze_feature_importance(df_with_features)

## Step 6: Model Selection

In this section, we will select and compare a variety of machine learning and deep learning models for the text classification task. The goal is to identify the best-performing model for classifying medical symptoms based on the engineered features. Both traditional ML algorithms and deep learning architectures will be considered. All steps are thoroughly commented for transparency and reproducibility.

In [ ]:
def train_and_evaluate_models(X, y):
    """
    Train and evaluate multiple machine learning models
    
    Args:
        X (sparse matrix): Input features
        y (array): Target labels
    
    Returns:
        dict: Model performance metrics
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Define models
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Naive Bayes': MultinomialNB(),
        'Support Vector Machine': SVC(probability=True),
        'Random Forest': RandomForestClassifier(),
        'Decision Tree': DecisionTreeClassifier()
    }
    
    # Results storage
    results = {}
    
    # Train and evaluate each model
    for name, model in models.items():
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5)
        
        # Fit model
        model.fit(X_train, y_train)
        
        # Predictions
        y_pred = model.predict(X_test)
        
        # Metrics
        results[name] = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, average='weighted'),
            'Recall': recall_score(y_test, y_pred, average='weighted'),
            'F1 Score': f1_score(y_test, y_pred, average='weighted'),
            'Cross-Val Score': cv_scores.mean()
        }
    
    return results

# Run model evaluation
model_results = train_and_evaluate_models(X, y)

# Display results
for model, metrics in model_results.items():
    print(f"\n{model} Performance:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

## Step 7: Model Training and Hyperparameter Optimization

This section covers splitting the data into training and testing sets, encoding and normalizing features, and performing hyperparameter optimization. We will use cross-validation to ensure robust model selection and avoid overfitting. All code is commented for clarity.

### Hyperparameter Optimization and Cross-Validation

To ensure robust model selection, we perform hyperparameter optimization using grid search and evaluate models with k-fold cross-validation (e.g., 5 or 10 folds). This process helps identify the best model configuration and prevents overfitting. All code is commented for transparency.

In [ ]:
# Hyperparameter optimization and cross-validation for Logistic Regression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Define parameter grid for Logistic Regression
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

# Initialize Logistic Regression model
logreg = LogisticRegression(max_iter=1000, random_state=42)

# Set up stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(logreg, param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_normalized, y_encoded)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validated F1 score: {grid_search.best_score_:.4f}")

# Use the best estimator for further evaluation
best_logreg = grid_search.best_estimator_

In [ ]:
def visualize_model_performance(results):
    """
    Visualize model performance metrics
    
    Args:
        results (dict): Model performance metrics
    """
    # Prepare data for plotting
    metrics_df = pd.DataFrame.from_dict(results, orient='index')
    
    # Plot
    plt.figure(figsize=(12, 6))
    metrics_df.plot(kind='bar', rot=45)
    plt.title('Model Performance Comparison')
    plt.xlabel('Models')
    plt.ylabel('Score')
    plt.tight_layout()
    plt.show()

# Visualize results
visualize_model_performance(model_results)

## Step 8: Model Evaluation and Tuning

We will evaluate the trained models using multiple metrics, including accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix. Cross-validation results will be reported, and the best model will be selected based on these metrics. If results are suboptimal, we will iterate and tune the models further.

### Model Evaluation Metrics and Confusion Matrix

We evaluate each model using a comprehensive set of metrics: accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix. This provides a holistic view of model performance and supports research transparency. All code is commented for clarity.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns

# Predict on the test set (using a new split for demonstration)
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
best_logreg.fit(X_train, y_train)
y_pred = best_logreg.predict(X_test)

# Compute evaluation metrics
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC-AUC (for multiclass, use 'ovr' strategy)
if len(label_encoder.classes_) > 2:
    y_score = best_logreg.predict_proba(X_test)
    roc_auc = roc_auc_score(y_test, y_score, multi_class='ovr')
    print(f"Multiclass ROC-AUC: {roc_auc:.4f}")
else:
    y_score = best_logreg.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_score)
    print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
def validate_research_hypothesis(results):
    """
    Validate research hypotheses based on model performance
    
    Args:
        results (dict): Model performance metrics
    """
    print("\n--- Research Hypothesis Validation ---")
    
    # Hypothesis threshold (suggest 0.75 as sufficient performance)
    threshold = 0.75
    
    # Check if text classification meets decision support criteria
    hypothesis_met = any(
        metrics['Precision'] > threshold and 
        metrics['Recall'] > threshold
        for metrics in results.values()
    )
    
    if hypothesis_met:
        print("H1a Supported: Text analysis provides sufficient precision and recall")
        print("Recommendation: Suitable for provider decision support")
    else:
        print("H10 Supported: Text analysis provides insufficient precision and recall")
        print("Recommendation: Further model refinement needed")

# Validate hypothesis
validate_research_hypothesis(model_results)

## Step 9: Research Hypothesis Validation

Based on the model evaluation results, we will validate the research hypotheses (H10/H1a) regarding the effectiveness of text-based classification for provider decision support. The decision will be based on whether the models achieve sufficient precision and recall.

## Research Limitations and Future Work
1. Limited to text-based classification
2. Potential bias in dataset
3. Future work: Integrate audio classification
4. Explore advanced deep learning architectures
5. Collect more diverse medical symptom data

## Conclusion
Comprehensive text classification workflow for medical symptom analysis, addressing research questions and hypotheses through systematic approach.

## Step 10: Research Limitations, Future Work, and Conclusion

This section summarizes the limitations of the current study, outlines directions for future research (including audio classification and advanced deep learning), and provides a final conclusion for the committee.